# 03 – Bevölkerung und Inzidenz

**Ziel:** Jeder Zeile der Meldedaten die Einwohnerzahl des Kreises zuordnen und daraus die **Inzidenz** berechnen (Fälle je 100.000 Einwohner). Erst damit sind große und kleine Kreise vergleichbar.

**Eingang:**
- `02_Daten/aufbereitet/meldedaten_kreis_woche.csv` (aus Notebook 01)
- `02_Daten/Bevölkerung/12411-01-01-4_flat.csv` (Regionaldatenbank, Bevölkerung am 31.12., alle Kreise)

**Ergebnis:** `02_Daten/aufbereitet/meldedaten_inzidenz.csv`

## Schritt 1: Bevölkerungsdatei einlesen

Die Datei ist ein „Flat-File“: eine Zeile pro Kreis, Stichtag und Geschlecht. Besonderheiten:
- Zeichensatz **Latin-1** (`encoding="latin-1"`), Trennzeichen **Semikolon**
- Wir behalten nur **„Insgesamt“** (nicht männlich/weiblich)
- Nur **5-stellige Kreisschlüssel** (AGS). Kürzere Schlüssel sind Bundesländer und Regierungsbezirke.
- Ein **„-“** statt einer Zahl heißt: diesen Kreis gab es zu dem Stichtag nicht (mehr), z. B. Kreise vor einer Gebietsreform. Diese Zeilen fallen weg.
- `dtype=str` liest alles als Text. Sonst würde aus dem Schlüssel „05315“ (Köln) die Zahl 5315, und die führende Null ginge verloren.

In [ ]:
from pathlib import Path
import re
import datetime as dt

import pandas as pd
import matplotlib.pyplot as plt

ORDNER = Path("../02_Daten")

roh = pd.read_csv(ORDNER / "Bevölkerung" / "12411-01-01-4_flat.csv", sep=";", encoding="latin-1", dtype=str)

bev = roh[(roh["2_variable_attribute_label"] == "Insgesamt")
          & (roh["1_variable_attribute_code"].str.len() == 5)
          & (roh["value"].str.isdigit())]
bev = pd.DataFrame({
    "AGS": bev["1_variable_attribute_code"],
    "Name_Destatis": bev["1_variable_attribute_label"],
    "Stichtag_Jahr": bev["time"].str[:4].astype(int),
    "Einwohner": bev["value"].astype(int),
})
print("Stichtage:", sorted(bev["Stichtag_Jahr"].unique()))
print("Kreise am 31.12.2025:", (bev["Stichtag_Jahr"] == 2025).sum())

## Schritt 2: Kreisnamen zuordnen

SurvStat und Destatis schreiben die Kreise unterschiedlich:

| SurvStat | Destatis |
|---|---|
| SK Köln | Köln, kreisfreie Stadt |
| LK Rhein-Kreis Neuss | Rhein-Kreis Neuss, Kreis |
| LK Euskirchen | Euskirchen, Kreis |

Die Funktion `kurzname()` entfernt die Zusätze („kreisfreie Stadt“, „Landkreis“, „Kreis“ …) und schreibt alles klein. Bleibt derselbe Name übrig **und** passt die Art (SK = kreisfreie Stadt, LK = Landkreis), gehören die beiden zusammen. Die Art ist wichtig, weil es z. B. „Kassel, kreisfreie Stadt“ und „Kassel, Landkreis“ gibt.

21 Kreise heißen so verschieden, dass das nicht klappt (z. B. „SK Halle“ und „Halle (Saale)“). Diese ordnen wir **von Hand** über den Kreisschlüssel zu. Ich habe jeden einzeln nachgeschlagen.

In [ ]:
def kurzname(name):
    name = re.sub(r",? ?(kreisfreie Stadt|Landkreis|Kreis|Stadtkreis|Wissenschaftsstadt|Universitätsstadt|Hansestadt|Stadt)\b", "", name)
    name = re.sub(r"^(Landkreis|Kreis|Stadtkreis) ", "", name)
    return re.sub(r"\s+", " ", name).strip(" ,").lower()

VON_HAND = {
    "LK Altenkirchen": "07132", "LK Bitburg-Prüm": "07232", "LK Landsberg a.Lech": "09181",
    "LK Lindau": "09776", "LK Sankt Wendel": "10046", "LK Stadtverband Saarbrücken": "10041",
    "SK Brandenburg a.d.Havel": "12051", "SK Frankenthal": "07311", "SK Freiburg i.Breisgau": "08311",
    "SK Halle": "15002", "SK Hamburg": "02000", "SK Kassel": "06611", "SK Kempten": "09763",
    "SK Landau i.d.Pfalz": "07313", "SK Ludwigshafen": "07314", "SK Magdeburg": "15003",
    "SK Mülheim a.d.Ruhr": "05117", "SK Neustadt a.d.Weinstraße": "07316", "SK Offenbach": "06413",
    "SK Oldenburg": "03403", "SK Wiesbaden": "06414",
}

meldedaten = pd.read_csv(ORDNER / "aufbereitet" / "meldedaten_kreis_woche.csv", parse_dates=["Wochenbeginn"])

aktuell = bev[bev["Stichtag_Jahr"] == 2025]
zuordnung = {}
for kreis in meldedaten["Kreis"].unique():
    if kreis in VON_HAND:
        zuordnung[kreis] = VON_HAND[kreis]
        continue
    art, rest = kreis.split(" ", 1)
    if art in ("SK", "LK"):
        stadt = aktuell["Name_Destatis"].str.contains("kreisfreie Stadt|Stadtkreis")
        passt_art = stadt == (art == "SK")
    else:                      # z. B. „Region Hannover“: ganzer Name, Art egal
        rest, passt_art = kreis, True
    treffer = aktuell[(aktuell["Name_Destatis"].map(kurzname) == kurzname(rest)) & passt_art]
    if len(treffer) == 1:
        zuordnung[kreis] = treffer["AGS"].item()

zuordnung = pd.Series(zuordnung, name="AGS").rename_axis("Kreis").reset_index()
print(len(zuordnung), "von", meldedaten["Kreis"].nunique(), "Kreisen zugeordnet")

## Schritt 3: Zuordnung prüfen

Drei Prüfungen, jede bricht bei einem Fehler ab:
1. **Alle 400** SurvStat-Kreise haben einen Schlüssel.
2. **Kein Schlüssel doppelt**: Zwei SurvStat-Kreise dürfen nicht auf denselben Destatis-Kreis zeigen.
3. **Stichprobe** mit bekannten Schlüsseln: Köln 05315, Düsseldorf 05111, Mainz 07315, Region Hannover 03241.

Danach siehst du unsere Städte mit beiden Namen nebeneinander, zum Gegenlesen.

In [ ]:
assert len(zuordnung) == 400
assert not zuordnung["AGS"].duplicated().any()
ags = dict(zip(zuordnung["Kreis"], zuordnung["AGS"]))
assert ags["SK Köln"] == "05315" and ags["SK Düsseldorf"] == "05111"
assert ags["SK Mainz"] == "07315" and ags["Region Hannover"] == "03241"
print("Alle Prüfungen bestanden")

namen = aktuell[["AGS", "Name_Destatis"]]
staedte = "Köln|Düsseldorf|Bonn|Aachen|Mainz|Mönchengladbach|Krefeld|Neuss|Hamburg|Berlin|Bremen|Hannover|Leipzig|Dresden|Nürnberg|Münster|Bielefeld"
zuordnung.merge(namen, on="AGS").query("Kreis.str.contains(@staedte)", engine="python")

## Schritt 4: Welche Einwohnerzahl gilt für welches Jahr?

Die Bevölkerung wird jeweils zum **31.12.** gezählt. Für die Inzidenz eines Jahres nehmen wir den **Stand vom 31.12. des Vorjahres**, also die Einwohner zu Jahresbeginn. Beispiel: Meldejahr 2024 → Bevölkerung am 31.12.2023.

**Zwei Lücken**, die in die Methodik gehören:
- Die Tabelle beginnt am **31.12.2016**. Für die Meldejahre **2015 und 2016** fehlt der Vorjahresstand. Wir nehmen dort ersatzweise den 31.12.2016. Die Bevölkerung ändert sich in einem Jahr meist nur um wenige Prozent. Das betrifft nur die ersten beiden Influenza-Jahre.
- **Zensus 2022:** Ab dem 31.12.2022 beruhen die Zahlen auf dem Zensus 2022, davor auf dem Zensus 2011. Der Zensus 2022 hat weniger Einwohner ergeben, als bis dahin fortgeschrieben waren. Die folgende Zelle zeigt die Summe über alle Kreise: Zwischen 2021 und 2022 siehst du keinen echten Rückgang der Bevölkerung, sondern diesen methodischen Sprung.

In [ ]:
(bev.groupby("Stichtag_Jahr")["Einwohner"].sum() / 1e6).round(2).rename("Einwohner in Mio.")

In [ ]:
bev["Meldejahr"] = bev["Stichtag_Jahr"] + 1
ersatz = bev[bev["Stichtag_Jahr"] == 2016].copy()
for jahr in [2015, 2016]:
    bev = pd.concat([bev, ersatz.assign(Meldejahr=jahr)])

# Für Meldejahre nach dem letzten Bevölkerungsstand gilt der letzte Stand (Fortschreibung erscheint mit Verzug)
letzter = bev[bev["Meldejahr"] == bev["Meldejahr"].max()]
for jahr in range(letzter["Meldejahr"].max() + 1, dt.date.today().year + 2):
    bev = pd.concat([bev, letzter.assign(Meldejahr=jahr)])

einwohner = zuordnung.merge(bev[["AGS", "Meldejahr", "Einwohner"]], on="AGS")
print("Meldejahre mit Einwohnern:", sorted(einwohner["Meldejahr"].unique()))

## Schritt 5: Inzidenz berechnen

**Inzidenz = Fälle ÷ Einwohner × 100.000**

`merge()` verbindet die beiden Tabellen über **Kreis und Meldejahr**, ähnlich wie SVERWEIS in Excel. Danach prüfen wir, dass keine Zeile ohne Einwohnerzahl geblieben ist und dass keine Zeile verloren ging.

In [ ]:
daten = meldedaten.merge(einwohner, on=["Kreis", "Meldejahr"], how="left")

assert len(daten) == len(meldedaten)
assert daten["Einwohner"].notna().all()

daten["Inzidenz"] = daten["Faelle"] / daten["Einwohner"] * 100_000
daten.head()

## Schritt 6: Jetzt vergleichbar

Dasselbe Diagramm wie in Notebook 01, aber als **Inzidenz**. Köln und Hamburg liegen jetzt auf derselben Skala.

In [ ]:
auswahl = daten[(daten["Erreger"] == "Influenza") & daten["Kreis"].isin(["SK Köln", "SK Hamburg"])]

fig, ax = plt.subplots(figsize=(12, 4))
for kreis, teil in auswahl.groupby("Kreis"):
    ax.plot(teil["Wochenbeginn"], teil["Inzidenz"], label=kreis, linewidth=1)
ax.set_ylabel("Fälle je 100.000 Einwohner und Woche")
ax.set_title("Influenza – Inzidenz pro Meldewoche")
ax.legend()
plt.show()

## Schritt 7: Ergebnis speichern

Die Tabelle aus Notebook 01 bekommt drei neue Spalten:

| Spalte | Bedeutung |
|---|---|
| AGS | amtlicher Kreisschlüssel (5 Stellen) |
| Einwohner | Bevölkerung am 31.12. des Vorjahres |
| Inzidenz | Fälle je 100.000 Einwohner in dieser Woche |

In [ ]:
datei_aus = ORDNER / "aufbereitet" / "meldedaten_inzidenz.csv"
daten.to_csv(datei_aus, index=False)
print("Gespeichert:", datei_aus, "|", len(daten), "Zeilen")

## Was du für die Methodik mitnehmen kannst

- Quelle: Statistische Ämter des Bundes und der Länder, Regionaldatenbank Deutschland, Tabelle 12411-01-01-4 (Fortschreibung des Bevölkerungsstandes, Stichtag 31.12.), Abruf 13.11.2026, Lizenz dl-de/by-2-0
- Bezugsbevölkerung: Stand 31.12. des Vorjahres; für 2015 und 2016 ersatzweise 31.12.2016
- Kreise über Namen zugeordnet (379 automatisch, 21 von Hand), alle 400 geprüft
- Bruch durch den Zensus 2022 ab Stichtag 31.12.2022